In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA gold;
SET TIME ZONE 'UTC';

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.gold.tmp_worlds
WITH window_data AS (
  SELECT MIN(source_file_date) AS snapshot_date
    FROM tibia_analytics.silver.worlds
   WHERE source_file_date > (
           SELECT COALESCE(MAX(last_seen_date), DATE '1997-01-07')
             FROM tibia_analytics.gold.worlds
         )
),
base AS (
  SELECT world_id,
         world AS world_name,
         location,
         pvp_type,
         premium_only,
         transfer_type,
         battleye_protected,
         battleye_date,
         world_type,
         tournament_type,
         source_file_date
    FROM tibia_analytics.silver.worlds
    WHERE source_file_date = (
           SELECT snapshot_date 
             FROM window_data
          )
)
SELECT world_id,
       world_name,
       location,
       pvp_type,
       premium_only,
       transfer_type,
       battleye_protected,
       battleye_date,
       world_type,
       tournament_type,
       source_file_date
  FROM base;

In [0]:
MERGE INTO tibia_analytics.gold.worlds AS target
USING tibia_analytics.gold.tmp_worlds AS source
ON target.world_id          = source.world_id
WHEN MATCHED THEN UPDATE SET
  target.world_name         = source.world_name,
  target.location           = source.location,
  target.pvp_type           = source.pvp_type,
  target.premium_only       = source.premium_only,
  target.transfer_type      = source.transfer_type,
  target.battleye_protected = source.battleye_protected,
  target.battleye_date      = source.battleye_date,
  target.world_type         = source.world_type,
  target.tournament_type    = source.tournament_type,
  target.last_seen_date     = source.source_file_date,
  target.is_active          = TRUE,
  target.updated_at         = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  world_id,
  world_name,
  location,
  pvp_type,
  premium_only,
  transfer_type,
  battleye_protected,
  battleye_date,
  world_type,
  tournament_type,
  first_seen_date,
  last_seen_date,
  is_active,
  created_at,
  updated_at
)
VALUES (
  source.world_id,
  source.world_name,
  source.location,
  source.pvp_type,
  source.premium_only,
  source.transfer_type,
  source.battleye_protected,
  source.battleye_date,
  source.world_type,
  source.tournament_type,
  source.source_file_date,
  source.source_file_date,
  TRUE,
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP()
);

In [0]:
UPDATE tibia_analytics.gold.worlds
   SET is_active  = FALSE,
       updated_at = CURRENT_TIMESTAMP()
 WHERE DATE_DIFF(
         (SELECT MAX(last_seen_date) FROM tibia_analytics.gold.worlds),
         last_seen_date
       ) > 7
   AND is_active  = TRUE;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.gold.tmp_worlds;